In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
import os

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, Dense, Input, LayerNormalization,
    MultiHeadAttention, Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.optimizers import Adam


# Task
Complete the air quality prediction task by enhancing the data cleaning process to ensure all missing values are handled by linear interpolation, forward-fill, and backward-fill, then re-prepare the data for time series modeling, re-build and re-train the neural network model, and finally re-evaluate its performance on the dataset from "/content/drive/MyDrive/air+quality/AirQualityUCI.csv", providing a summary of the model's performance and key insights.

## Re-run Data Cleaning with Enhanced NaN Handling

### Subtask:
Ensure all missing values are handled by linear interpolation, forward-fill, and backward-fill after initial data cleaning and before further processing.


**Reasoning**:
The first instruction requires re-executing a series of previous code cells to reset the DataFrame `df` to a specific state, which is essential for applying the enhanced NaN handling process from the beginning. This first code block reloads the dataset.



In [51]:
DATASET_PATH = r"/content/drive/MyDrive/air+quality/AirQualityUCI.csv"
df = pd.read_csv(DATASET_PATH, sep=';', decimal=',', encoding='iso-8859-1')
display(df.head())

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


**Reasoning**:
Following the first instruction, the next step is to re-execute cell '4a5b1091', which replaces all instances of '-200.0' with `np.nan` in the DataFrame to standardize missing value representation.



In [52]:
df.replace(-200.0, np.nan, inplace=True)
print("Replaced -200.0 with NaN values.")

Replaced -200.0 with NaN values.


**Reasoning**:
The next step is to re-execute cell '093f86d0', which converts the 'Date' column to datetime objects, as specified in the instructions for resetting the DataFrame.



In [53]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
print("Converted 'Date' column to datetime objects.")

Converted 'Date' column to datetime objects.


**Reasoning**:
The next step is to re-execute cell '53d82077', which combines the 'Date' and 'Time' columns into a single 'DateTime' column and drops the original 'Time' column, as specified in the instructions for resetting the DataFrame.



In [54]:
df['DateTime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], format='%Y-%m-%d %H.%M.%S')
df.drop('Time', axis=1, inplace=True)
print("Converted 'Time' column to datetime objects and combined with 'Date' into a new 'DateTime' column. Original 'Time' column dropped.")

Converted 'Time' column to datetime objects and combined with 'Date' into a new 'DateTime' column. Original 'Time' column dropped.


**Reasoning**:
The next step is to re-execute cell '86f4efe2', which drops the unnecessary columns 'Unnamed: 15' and 'Unnamed: 16', as specified in the instructions for resetting the DataFrame.



In [55]:
df.drop(columns=['Unnamed: 15', 'Unnamed: 16'], errors='ignore', inplace=True)
print("Dropped 'Unnamed: 15' and 'Unnamed: 16' columns if they existed.")

Dropped 'Unnamed: 15' and 'Unnamed: 16' columns if they existed.


**Reasoning**:
Now that the DataFrame has been reset and unnecessary columns dropped, the next step is to sort by 'DateTime', set it as the index, apply linear interpolation, then forward-fill and backward-fill for robust NaN handling, and finally reset the index, as per instructions 2 through 6 of the subtask.



In [56]:
df.sort_values(by='DateTime', inplace=True)
df.set_index('DateTime', inplace=True)
df.interpolate(method='linear', limit_direction='both', inplace=True)
df.ffill(inplace=True) # Fill any remaining NaNs at the beginning
df.bfill(inplace=True) # Fill any remaining NaNs at the end
df.reset_index(inplace=True)
print("Missing values handled using linear interpolation, forward-fill, and backward-fill.")

Missing values handled using linear interpolation, forward-fill, and backward-fill.


**Reasoning**:
After applying the enhanced NaN handling methods, the final step in this subtask is to verify that all missing values have indeed been handled by printing the sum of null values for each column in the DataFrame, as per instruction 7 of the subtask.



In [57]:
print('Missing values after enhanced cleaning:')
print(df.isnull().sum())

Missing values after enhanced cleaning:
DateTime         114
Date               0
CO(GT)             0
PT08.S1(CO)        0
NMHC(GT)           0
C6H6(GT)           0
PT08.S2(NMHC)      0
NOx(GT)            0
PT08.S3(NOx)       0
NO2(GT)            0
PT08.S4(NO2)       0
PT08.S5(O3)        0
T                  0
RH                 0
AH                 0
dtype: int64


**Reasoning**:
The previous output showed that the 'DateTime' column still contains 114 missing values (NaTs) even after applying interpolation, forward-fill, and backward-fill. Since 'DateTime' is a critical column for time series analysis and these NaTs prevent proper chronological ordering and model preparation, the next logical step to ensure 'all missing values are handled' (as per the subtask) is to drop the rows where 'DateTime' is NaN.



In [58]:
df.dropna(subset=['DateTime'], inplace=True)
print("Dropped rows where 'DateTime' is NaN.")
print('Missing values after dropping DateTime NaNs:')
print(df.isnull().sum())

Dropped rows where 'DateTime' is NaN.
Missing values after dropping DateTime NaNs:
DateTime         0
Date             0
CO(GT)           0
PT08.S1(CO)      0
NMHC(GT)         0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
dtype: int64


## Re-prepare Data for Time Series Modeling

### Subtask:
Re-structure the now fully cleaned and scaled data into sequences (X and y) and re-split them into training and testing sets, using the corrected DataFrame.


**Reasoning**:
The previous steps involved cleaning and handling missing values. Now, as per the subtask instruction, I need to extract 'hour' and 'day_of_week' from the 'DateTime' column and add them as new columns to the DataFrame `df`. This is the first step in re-preparing the data for time series modeling.



In [59]:
df['hour'] = df['DateTime'].dt.hour
df['day_of_week'] = df['DateTime'].dt.dayofweek
print("Extracted 'hour' and 'day_of_week' from 'DateTime' column.")

Extracted 'hour' and 'day_of_week' from 'DateTime' column.


**Reasoning**:
Now that time-based features ('hour' and 'day_of_week') have been extracted, the next step is to identify all numerical columns in the updated DataFrame, instantiate a new `StandardScaler` object, and apply it to these numerical features as per instructions 2, 3, and 4 of the subtask. This ensures all relevant features are scaled before sequence creation.



In [60]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
# The 'Date' column is of datetime type and 'DateTime' is also datetime, so they won't be included by select_dtypes(include=np.number).

print(f"Numerical columns to be scaled: {numerical_cols}")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("Numerical features scaled using StandardScaler.")

Numerical columns to be scaled: ['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'hour', 'day_of_week']
Numerical features scaled using StandardScaler.


**Reasoning**:
Now that the numerical features have been scaled, the next step is to re-apply the `create_sequences` function to the fully cleaned and scaled DataFrame `df`. It's important to ensure that the `FEATURES` variable is updated to reflect the `numerical_cols` identified, which now include 'hour' and 'day_of_week', before generating the input sequences `X` and target sequences `y`.



In [61]:
FEATURES = numerical_cols
X, y = create_sequences(df, TARGET_COLUMN, look_back_window, forecast_horizon, FEATURES)

print(f"Shape of X (input sequences): {X.shape}")
print(f"Shape of y (target sequences): {y.shape}")

Shape of X (input sequences): (9333, 24, 15)
Shape of y (target sequences): (9333, 1)


**Reasoning**:
Now that the `X` and `y` arrays have been generated from the fully cleaned and scaled data, the final step in this subtask is to split them into training and testing sets. For time series data, it is crucial to perform a chronological split to avoid data leakage, typically using the first 80% for training and the remaining 20% for testing.



In [62]:
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, y_train = X[:split_index], y[:split_index]
X_test, y_test = X[split_index:], y[split_index:]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (7466, 24, 15)
y_train shape: (7466, 1)
X_test shape: (1867, 24, 15)
y_test shape: (1867, 1)


## Re-build and Re-compile the Model

### Subtask:
Re-construct and re-compile the neural network model with the same architecture and settings to prepare for re-training with the corrected data.


**Reasoning**:
To re-build and re-compile the neural network model as required by the subtask, I will define the input shape, construct a Sequential model with an Input layer, an LSTM layer, and a Dense output layer, then compile it with the specified optimizer and loss function, and finally print the model summary.



In [63]:
input_shape = X_train.shape[1:] # (look_back_window, num_features)

model = Sequential([
    Input(shape=input_shape), # Explicit Input layer to avoid UserWarning
    LSTM(50, activation='relu', return_sequences=False),
    Dense(forecast_horizon) # Output layer for predicting the next 'forecast_horizon' steps
])

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])

model.summary()
print("Neural network model constructed and compiled.")

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 50)             │        13,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,251 (51.76 KB)

 Trainable params: 13,251 (51.76 KB)

 Non-trainable params: 0 (0.00 B)

Neural network model constructed and compiled.


**Reasoning**:
The model has been successfully constructed and compiled. The next logical step is to train the model using the prepared training data (`X_train`, `y_train`). This aligns with the overall goal of developing and training a neural network model for time series forecasting.



In [64]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)
print("Model training complete.")

Epoch 1/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 0.4272 - mean_absolute_error: 0.4730 - val_loss: 0.3478 - val_mean_absolute_error: 0.4217
Epoch 2/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.1970 - mean_absolute_error: 0.3107 - val_loss: 0.3022 - val_mean_absolute_error: 0.3983
Epoch 3/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.1668 - mean_absolute_error: 0.2847 - val_loss: 0.2850 - val_mean_absolute_error: 0.3893
Epoch 4/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.1489 - mean_absolute_error: 0.2665 - val_loss: 0.2903 - val_mean_absolute_error: 0.3911
Epoch 5/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.1473 - mean_absolute_error: 0.2629 - val_loss: 0.2750 - val_mean_absolute_error: 0.3751
Epoch 6/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 0.1374 - mean_absolute_error: 0.2513 - val_loss: 0.2719 - val_mean_absolute_error: 0.3712
Epoch 7/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.1305 - mean_absolute_error: 0.2